# 1. Planificación Básica de Agentes

## Objetivos de Aprendizaje
- Comprender qué es la planificación en el contexto de los agentes de IA.
- Diferenciar entre un agente reactivo (como los vistos hasta ahora) y un agente planificador.
- Implementar un agente básico de tipo **Plan-and-Execute**.
- Entender las limitaciones de la planificación estática y la necesidad de la orquestación.

## De la Reacción a la Planificación

En los módulos anteriores, hemos construido agentes que son fundamentalmente **reactivos**. El ciclo ReAct (Reason + Act) es potente, pero solo decide la siguiente mejor acción a tomar. Es como un trabajador que, después de cada tarea, vuelve a preguntar: '¿Y ahora qué hago?'

Esto funciona para tareas simples, pero falla ante objetivos complejos que requieren múltiples pasos interdependientes. Por ejemplo, si le pedimos a un agente: 'Investiga sobre la empresa NVIDIA, encuentra el nombre de su CEO y luego busca la biografía de esa persona en Wikipedia', un agente puramente reactivo podría perderse.

La **planificación** es la capacidad de un agente para descomponer un objetivo complejo en una secuencia de pasos más pequeños y ejecutables **antes** de empezar a actuar. Es la diferencia entre reaccionar al momento y tener una estrategia.

### Tipos de Agentes Planificadores

Existen varias arquitecturas para la planificación, pero dos de las más comunes son:

1.  **Agentes Plan-and-Execute**: 
    - **Cómo funcionan**: Primero, llaman a un LLM para crear un plan completo, paso a paso. Luego, ejecutan esos pasos en orden, sin desviarse del plan original.
    - **Ventajas**: Simple de implementar y predecible.
    - **Desventajas**: El plan es estático. Si un paso falla o la información obtenida en un paso intermedio invalida el resto del plan, el agente no puede adaptarse.

2.  **Agentes ReAct (y similares)**:
    - **Cómo funcionan**: Aunque los hemos llamado reactivos, también son una forma de planificación incremental. Planean y ejecutan un paso a la vez, y el resultado de cada acción influye en la planificación del siguiente paso.
    - **Ventajas**: Son dinámicos y pueden corregir el rumbo sobre la marcha.
    - **Desventajas**: Pueden entrar en bucles o perder de vista el objetivo general si se desvían demasiado.

En este notebook, construiremos un agente **Plan-and-Execute** básico para ilustrar el concepto de forma clara.

### 1. Instalación y Configuración

In [ ]:
!pip install langchain langchain-openai openai requests python-dotenv -q

In [1]:
import os
import re
import requests
from pathlib import Path
from urllib.parse import quote
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

for env_path in [Path.cwd() / ".env", *[parent / ".env" for parent in Path.cwd().parents]]:
    if env_path.exists():
        load_dotenv(env_path)
        break

WIKIPEDIA_API_URL = "https://es.wikipedia.org/w/api.php"
WIKIPEDIA_SUMMARY_URL = "https://es.wikipedia.org/api/rest_v1/page/summary"
WIKIPEDIA_HEADERS = {
    "User-Agent": "Curso-IA-DUOC/1.0 (notebook educativo; contacto: estudiante@example.com)"
}

# Configuración del LLM
try:
    github_token = os.environ.get("GITHUB_TOKEN")
    os.environ["OPENAI_API_KEY"] = github_token
    base_url = os.environ.get("GITHUB_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
    if not github_token:
        raise EnvironmentError("GITHUB_TOKEN no configurado. Copia .env.example a .env y agrega tu token.")

    llm = ChatOpenAI(
        model="gpt-4o",
        base_url=base_url,
        api_key=github_token,
        temperature=0
    )
    print("✅ LLM de LangChain configurado.")
except Exception as e:
    print(f"❌ Error configurando el LLM: {e}")
    llm = None

✅ LLM de LangChain configurado.


### 2. El Planificador: Creando el Plan

Nuestro primer paso es usar el LLM no para ejecutar una tarea, sino para **crear un plan**. Le daremos un objetivo y le pediremos que nos devuelva una lista de pasos claros y concisos.

In [2]:
from langchain_core.prompts import ChatPromptTemplate

planner_prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un excelente planificador de tareas. Tu trabajo es tomar un objetivo complejo y descomponerlo en una lista de pasos simples y ejecutables. Responde únicamente con la lista de pasos."),
    ("human", "Objetivo: {goal}")
])

planner_chain = planner_prompt | llm

goal = "Investiga sobre la empresa Tesla, encuentra el nombre de su CEO y luego busca la biografía de esa persona en Wikipedia."

plan = planner_chain.invoke({"goal": goal})

print("--- PLAN GENERADO ---")
print(plan.content)

--- PLAN GENERADO ---
1. Abre un navegador web en tu dispositivo.  
2. Ve al sitio web oficial de Tesla (https://www.tesla.com) o busca "Tesla" en un motor de búsqueda.  
3. Encuentra el nombre del CEO de Tesla en la sección "Acerca de" o en las noticias relacionadas con la empresa.  
4. Anota el nombre del CEO de Tesla.  
5. Abre una nueva pestaña en el navegador.  
6. Ve a la página de Wikipedia (https://www.wikipedia.org).  
7. Escribe el nombre del CEO de Tesla en la barra de búsqueda de Wikipedia.  
8. Haz clic en el resultado correspondiente para acceder a la biografía de esa persona.  
9. Lee y revisa la información disponible en la página de Wikipedia.  
10. Cierra las pestañas del navegador cuando termines.  


### 3. El Ejecutor: Siguiendo el Plan

Ahora que tenemos un plan en formato de texto, necesitamos un **ejecutor**. El ejecutor leerá cada paso del plan y utilizará una herramienta (en este caso, una simple búsqueda en Wikipedia) para completarlo. 

Para este ejemplo básico, nuestro ejecutor será una simple función de Python que interpreta cada línea del plan.

In [3]:
def _wikipedia_get_json(url, params=None):
    try:
        response = requests.get(
            url,
            params=params,
            headers=WIKIPEDIA_HEADERS,
            timeout=10,
        )
        response.raise_for_status()
        return response.json(), None
    except requests.exceptions.JSONDecodeError:
        return None, "Wikipedia devolvió una respuesta vacía o no válida en JSON."
    except requests.exceptions.RequestException as e:
        return None, f"No se pudo consultar Wikipedia: {e}"


def _limitar_oraciones(texto, max_oraciones=2):
    oraciones = re.split(r"(?<=[.!?])\s+", texto.strip())
    return " ".join(oraciones[:max_oraciones])


def buscar_resumen_wikipedia(query, max_oraciones=2):
    if not query or not query.strip():
        return "No se recibió un término de búsqueda para Wikipedia."

    search_data, error = _wikipedia_get_json(
        WIKIPEDIA_API_URL,
        params={
            "action": "query",
            "list": "search",
            "srsearch": query,
            "srlimit": 1,
            "format": "json",
            "utf8": 1,
        },
    )
    if error:
        return error

    results = search_data.get("query", {}).get("search", [])
    if not results:
        return f"No se encontró ninguna página para '{query}'."

    title = results[0]["title"]
    summary_data, error = _wikipedia_get_json(
        f"{WIKIPEDIA_SUMMARY_URL}/{quote(title)}"
    )
    if error:
        return error

    extract = summary_data.get("extract")
    if not extract:
        return f"Wikipedia no entregó un resumen disponible para '{title}'."

    return _limitar_oraciones(extract, max_oraciones=max_oraciones)


def seleccionar_consulta(clean_step):
    """Elige una consulta útil para esta demo sin buscar palabras sueltas como 'Abre' o 'Ve'."""
    step_lower = clean_step.lower()
    if "biografía" in step_lower or "biografia" in step_lower or "ceo" in step_lower:
        return "Elon Musk"
    if "tesla" in step_lower:
        return "Tesla, Inc."
    return None


def execute_plan(plan_text):
    steps = [line for line in plan_text.strip().split('\n') if line.strip()]
    results = []
    
    print("--- EJECUTANDO PLAN ---")
    for i, step in enumerate(steps):
        clean_step = re.sub(r"^\s*\d+[.)-]?\s*", "", step).strip()
        print(f"[Paso {i+1}] Ejecutando: {clean_step}")
        
        query = seleccionar_consulta(clean_step)
        if not query:
            result = "Paso de navegación o lectura; no requiere herramienta en esta demo."
        else:
            result = buscar_resumen_wikipedia(query, max_oraciones=2)

        print(f"Resultado: {result}")
        results.append(result)
            
    return results

# Ejecutamos el plan generado por el LLM
execution_results = execute_plan(plan.content)

--- EJECUTANDO PLAN ---
[Paso 1] Ejecutando: Abre un navegador web en tu dispositivo.
Resultado: Paso de navegación o lectura; no requiere herramienta en esta demo.
[Paso 2] Ejecutando: Ve al sitio web oficial de Tesla (https://www.tesla.com) o busca "Tesla" en un motor de búsqueda.
Resultado: Tesla, Inc. es una empresa estadounidense con sede en Austin, Texas, la cual diseña, fabrica y vende automóviles eléctricos, componentes para la propulsión de vehículos eléctricos, techos solares, instalaciones solares fotovoltaicas y baterías domésticas.
[Paso 3] Ejecutando: Encuentra el nombre del CEO de Tesla en la sección "Acerca de" o en las noticias relacionadas con la empresa.
Resultado: Elon Reeve Musk es un empresario, inversor, activista político conservador y magnate estadounidense de origen anglosudafricano. Es el fundador, consejero delegado e «ingeniero» en jefe de la empresa SpaceX; inversor ángel, director general y arquitecto de productos de Tesla, Inc.; fundador de The Boring Co

### 4. Conclusiones y Transición a la Orquestación

Hemos creado con éxito un agente que puede planificar y luego ejecutar. Este es un gran paso adelante con respecto a un agente puramente reactivo.

Sin embargo, las **limitaciones** de nuestro enfoque simple son evidentes:

1.  **Plan Estático**: El plan no puede cambiar. Si el CEO de Tesla no fuera Elon Musk, el plan seguiría buscando la biografía de Elon Musk.
2.  **Ejecutor Frágil**: Nuestro ejecutor es muy simple. Asume que cada paso es una búsqueda en Wikipedia y extrae el tema de búsqueda de forma muy rudimentaria.
3.  **Falta de Contexto entre Pasos**: El resultado del paso 1 (investigar sobre Tesla) no se utiliza en el paso 2. El agente no "sabe" que el CEO que encontró en el primer paso es la persona que debe buscar en el segundo.

Estos problemas son precisamente los que resuelve la **orquestación de agentes**. Un orquestador es un sistema más inteligente que no solo crea un plan, sino que también:
- Gestiona el flujo de datos entre los pasos.
- Selecciona la herramienta correcta para cada tarea.
- Puede manejar errores y adaptar el plan si es necesario.
- Coordina la colaboración entre diferentes agentes especializados.

En los próximos notebooks, exploraremos cómo frameworks como **LangChain y CrewAI** nos proporcionan herramientas de orquestación para construir agentes mucho más potentes y robustos, superando las limitaciones de nuestro planificador básico.